# Action Definition Analysis: Merging Incremental Operator Actions

## Problem Statement

Currently, each operator action (CHANGE event) is treated as an independent datapoint. However, operators often take actions **incrementally** - for example, opening a valve slowly in multiple small steps over a short time window rather than making one large change.

**Example**: To move a setpoint from 50 to 60, an operator might:
- t=0: 50 → 52 (+2)
- t=30s: 52 → 54 (+2)
- t=1m: 54 → 57 (+3)
- t=2m: 57 → 60 (+3)

In the current similarity approach, these 4 actions are stored independently in the lookup table. At runtime, when the system finds a similar context, it might recommend just one of these steps (+2 or +3) instead of the full cumulative move (+10).

## Proposed Solution

1. **Analyze temporal patterns** - Study the distribution of time gaps between consecutive actions on the same tag within episodes
2. **Define a time threshold** - Below which consecutive actions on the same tag are considered part of the same "action sequence"  
3. **Merge action sequences** - Into composite actions representing the operator's full intent:
   - Net magnitude = final_value - initial_prev_value
   - Timestamp = first action in sequence
   - Direction = based on net change
4. **Rebuild the lookup table** with composite actions and compare with the original approach

## Step 1: Load Data

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm import tqdm

# Load core datasets
episodes_operated_tags_df = pd.read_excel('/home/h604827/ControlActions/RESULTS/episode_all_operator_action_plots/episodes_all_with_actions_and_deviations.xlsx')
ssd_df = pd.read_excel('/home/h604827/ControlActions/DATA/SSD_1071_SSD_output_1071_7Jan2026.xlsx')
events_df = pd.read_csv('/home/h604827/ControlActions/DATA/trip_filtered_events.csv')
pv_op_data_df = pd.read_parquet('/home/h604827/ControlActions/DATA/03LIC_1071_JAN_2026_filtered.parquet')
operating_limits_df = pd.read_csv('/home/h604827/ControlActions/DATA/operating_limits.csv')
benchmark_df = pd.read_excel('/home/h604827/ControlActions/RESULTS/alarm_episodes_benchmark.xlsx')

# Parse timestamps
events_df['VT_Start'] = pd.to_datetime(events_df['VT_Start'])
ssd_df['AlarmStart_rounded_minutes'] = pd.to_datetime(ssd_df['AlarmStart_rounded_minutes'])
ssd_df['AlarmEnd_rounded_minutes'] = pd.to_datetime(ssd_df['AlarmEnd_rounded_minutes'])
ssd_df['Tag_First_Transition_Start_minutes'] = pd.to_datetime(ssd_df['Tag_First_Transition_Start_minutes'])
episodes_operated_tags_df['AlarmStart'] = pd.to_datetime(episodes_operated_tags_df['AlarmStart'])
episodes_operated_tags_df['AlarmEnd'] = pd.to_datetime(episodes_operated_tags_df['AlarmEnd'])

# Filter no-deviation episodes
no_deviation_episode_ids = benchmark_df[benchmark_df['Hasdeviation'] == False]['EpisodeID'].tolist()

# PV tags for context
context_pv_tags = [col for col in pv_op_data_df.columns if col.endswith('.PV')]

print(f"Total episodes: {len(episodes_operated_tags_df)}")
print(f"No-deviation episodes: {len(no_deviation_episode_ids)}")
print(f"Events: {len(events_df):,}")
print(f"PV tags for context: {len(context_pv_tags)}")

Total episodes: 609
No-deviation episodes: 404
Events: 1,600,520
PV tags for context: 28


### Helper functions (reused from similarity approach notebook)

In [2]:
def get_deviation_start_for_episode(episode_id, alarm_start, alarm_end):
    """Get the deviation start time for an episode."""
    episode_ssd = ssd_df[ssd_df['AlarmStart_rounded_minutes'] == alarm_start]
    if len(episode_ssd) == 0:
        episode_ssd = ssd_df[
            (abs((ssd_df['AlarmStart_rounded_minutes'] - alarm_start).dt.total_seconds()) <= 60)
        ]
    if len(episode_ssd) == 0:
        return alarm_start - pd.Timedelta(minutes=30)
    target_ssd = episode_ssd[episode_ssd['TagName'] == '03LIC_1071']
    if len(target_ssd) > 0:
        return target_ssd['Tag_First_Transition_Start_minutes'].iloc[0]
    return episode_ssd['Tag_First_Transition_Start_minutes'].min()


def get_operator_actions_for_episode(alarm_start, alarm_end, target_sources=['03LIC_1071', '03LIC_1016', '03PIC_1013']):
    """Get all CHANGE events (operator actions) for target sources during an episode."""
    deviation_start = get_deviation_start_for_episode(None, alarm_start, alarm_end)
    actions = events_df[
        (events_df['ConditionName'] == 'CHANGE') &
        (events_df['Source'].isin(target_sources)) &
        (events_df['VT_Start'] >= deviation_start) &
        (events_df['VT_Start'] <= alarm_end)
    ].copy()
    if 'Description' in actions.columns:
        actions = actions[actions['Description'].isin(['SP', 'OP'])].copy()
    return actions, deviation_start


def get_pv_at_timestamp(timestamp, pv_tag):
    """Get PV value at or nearest to the given timestamp."""
    try:
        if hasattr(timestamp, 'tzinfo') and timestamp.tzinfo is not None:
            timestamp = timestamp.tz_localize(None)
        if timestamp in pv_op_data_df.index:
            return pv_op_data_df.loc[timestamp, pv_tag]
        idx = pv_op_data_df.index.get_indexer([timestamp], method='ffill')[0]
        if 0 <= idx < len(pv_op_data_df):
            return pv_op_data_df.iloc[idx][pv_tag]
        idx = pv_op_data_df.index.get_indexer([timestamp], method='bfill')[0]
        if 0 <= idx < len(pv_op_data_df):
            return pv_op_data_df.iloc[idx][pv_tag]
        return np.nan
    except Exception:
        return np.nan

print("Helper functions defined.")

Helper functions defined.


---
## Step 2: Analyze Temporal Patterns of Consecutive Actions

Before merging actions, we need to understand the **distribution of time gaps** between consecutive actions on the same tag within each episode. This will help us determine a reasonable threshold for grouping actions into sequences.

**Key questions:**
- How often do operators take multiple actions on the same tag in rapid succession?
- What does the time gap distribution look like? Is there a natural cutoff?
- Does this pattern differ across the three target tags?

In [3]:
# Collect all target-tag actions across all episodes with their time gaps
target_sources = ['03LIC_1071', '03LIC_1016', '03PIC_1013']
all_episodes_df = episodes_operated_tags_df.copy()

all_action_records = []

for _, episode in tqdm(all_episodes_df.iterrows(), total=len(all_episodes_df), desc="Collecting actions"):
    episode_id = episode['EpisodeID']
    alarm_start = episode['AlarmStart']
    alarm_end = episode['AlarmEnd']
    
    actions, deviation_start = get_operator_actions_for_episode(alarm_start, alarm_end, target_sources)
    
    if len(actions) == 0:
        continue
    
    # Clean: keep only rows with valid PrevValue and deduplicate
    actions_clean = actions.dropna(subset=['PrevValue']).copy()
    actions_clean = actions_clean.drop_duplicates(subset=['VT_Start', 'Source', 'Value'])
    
    # Compute magnitude
    actions_clean['Value_num'] = pd.to_numeric(actions_clean['Value'], errors='coerce')
    actions_clean['PrevValue_num'] = pd.to_numeric(actions_clean['PrevValue'], errors='coerce')
    actions_clean = actions_clean.dropna(subset=['Value_num', 'PrevValue_num'])
    actions_clean['magnitude'] = actions_clean['Value_num'] - actions_clean['PrevValue_num']
    
    for source in target_sources:
        source_actions = actions_clean[actions_clean['Source'] == source].sort_values('VT_Start')
        if len(source_actions) == 0:
            continue
        
        for i, (_, act) in enumerate(source_actions.iterrows()):
            record = {
                'episode_id': episode_id,
                'source': source,
                'timestamp': act['VT_Start'],
                'value': act['Value_num'],
                'prev_value': act['PrevValue_num'],
                'magnitude': act['magnitude'],
                'description': act.get('Description', None),
                'action_index_in_episode': i
            }
            all_action_records.append(record)

all_actions_df = pd.DataFrame(all_action_records)
print(f"Total individual actions collected: {len(all_actions_df)}")
print(f"\nActions per tag:")
print(all_actions_df['source'].value_counts())

Total individual actions collected: 3854

Actions per tag:
source
03PIC_1013    2406
03LIC_1071    1040
03LIC_1016     408
Name: count, dtype: int64


In [4]:
# Compute time gaps between consecutive actions on the SAME TAG within each episode
time_gaps = []

for (ep_id, source), group in all_actions_df.groupby(['episode_id', 'source']):
    group_sorted = group.sort_values('timestamp')
    if len(group_sorted) < 2:
        continue
    
    timestamps = group_sorted['timestamp'].values
    for i in range(1, len(timestamps)):
        gap_seconds = (pd.Timestamp(timestamps[i]) - pd.Timestamp(timestamps[i-1])).total_seconds()
        time_gaps.append({
            'episode_id': ep_id,
            'source': source,
            'gap_seconds': gap_seconds,
            'gap_minutes': gap_seconds / 60.0,
            'action_count_in_episode': len(group_sorted)
        })

gaps_df = pd.DataFrame(time_gaps)
print(f"Total consecutive time gaps computed: {len(gaps_df)}")
print(f"\nTime gap statistics (seconds):")
print(gaps_df['gap_seconds'].describe())
print(f"\nTime gap statistics (minutes):")
print(gaps_df['gap_minutes'].describe())

Total consecutive time gaps computed: 3599

Time gap statistics (seconds):
count     3599.000000
mean       185.641646
std        605.812892
min          0.547400
25%          1.253800
50%          5.478500
75%         82.650150
max      10201.649800
Name: gap_seconds, dtype: float64

Time gap statistics (minutes):
count    3599.000000
mean        3.094027
std        10.096882
min         0.009123
25%         0.020897
50%         0.091308
75%         1.377503
max       170.027497
Name: gap_minutes, dtype: float64


In [19]:
# Visualize the time gap distribution
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Time Gap Distribution (All Tags, 0-10 min)',
        'Time Gap Distribution by Tag (0-10 min)',
        'Cumulative Distribution of Time Gaps',
        'Time Gap Distribution (0-60 sec zoom)'
    ]
)

# Plot 1: Overall histogram (0-10 minutes)
gaps_under_10min = gaps_df[gaps_df['gap_minutes'] <= 10]['gap_minutes']
fig.add_trace(
    go.Histogram(x=gaps_under_10min, nbinsx=100, name='All Tags', marker_color='steelblue'),
    row=1, col=1
)

# Plot 2: Per-tag histograms (0-10 minutes)
colors = {'03LIC_1071': 'blue', '03LIC_1016': 'green', '03PIC_1013': 'orange'}
for source, color in colors.items():
    tag_gaps = gaps_df[(gaps_df['source'] == source) & (gaps_df['gap_minutes'] <= 10)]['gap_minutes']
    fig.add_trace(
        go.Histogram(x=tag_gaps, nbinsx=60, name=source, marker_color=color, opacity=0.6),
        row=1, col=2
    )

# Plot 3: CDF of time gaps
sorted_gaps = np.sort(gaps_df['gap_minutes'].values)
cdf = np.arange(1, len(sorted_gaps) + 1) / len(sorted_gaps)
fig.add_trace(
    go.Scatter(x=sorted_gaps, y=cdf, mode='lines', name='CDF', line=dict(color='darkblue')),
    row=2, col=1
)
# Add reference lines at candidate thresholds
for threshold in [1, 2, 3, 5]:
    pct_below = np.mean(gaps_df['gap_minutes'] <= threshold) * 100
    fig.add_vline(x=threshold, line_dash="dash", line_color="red", row=2, col=1,
                  annotation_text=f"{threshold}min: {pct_below:.0f}%")

# Plot 4: Zoomed histogram (0-60 seconds)
gaps_under_60s = gaps_df[gaps_df['gap_seconds'] <= 60]['gap_seconds']
fig.add_trace(
    go.Histogram(x=gaps_under_60s, nbinsx=60, name='<60s', marker_color='coral'),
    row=2, col=2
)

fig.update_xaxes(title_text="Time Gap (minutes)", row=1, col=1)
fig.update_xaxes(title_text="Time Gap (minutes)", row=1, col=2)
fig.update_xaxes(title_text="Time Gap (minutes)", row=2, col=1)
fig.update_xaxes(title_text="Time Gap (seconds)", row=2, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_yaxes(title_text="Cumulative Proportion", row=2, col=1)
fig.update_yaxes(title_text="Count", row=2, col=2)

fig.update_layout(height=700, title_text="Time Gap Distribution Between Consecutive Actions on Same Tag", barmode='overlay')
fig.show()

In [6]:
# Quantile analysis to help choose threshold
print("=== Time Gap Quantile Analysis (minutes) ===\n")
quantiles = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
print("Overall:")
for q in quantiles:
    val = gaps_df['gap_minutes'].quantile(q)
    print(f"  P{int(q*100):2d}: {val:.2f} min")

print()
for source in target_sources:
    tag_gaps = gaps_df[gaps_df['source'] == source]['gap_minutes']
    if len(tag_gaps) == 0:
        continue
    print(f"{source} ({len(tag_gaps)} gaps):")
    for q in [0.25, 0.50, 0.75, 0.90]:
        val = tag_gaps.quantile(q)
        print(f"  P{int(q*100):2d}: {val:.2f} min")
    print()

# Show what % of gaps fall below candidate thresholds
print("=== Percentage of gaps below candidate thresholds ===")
for threshold_min in [0.5, 1, 2, 3, 5, 10]:
    pct = (gaps_df['gap_minutes'] <= threshold_min).mean() * 100
    count = (gaps_df['gap_minutes'] <= threshold_min).sum()
    print(f"  <= {threshold_min:4.1f} min: {pct:5.1f}% ({count}/{len(gaps_df)} gaps)")

=== Time Gap Quantile Analysis (minutes) ===

Overall:
  P10: 0.02 min
  P20: 0.02 min
  P30: 0.02 min
  P40: 0.03 min
  P50: 0.09 min
  P60: 0.35 min
  P70: 0.87 min
  P80: 2.10 min
  P90: 6.86 min
  P95: 15.80 min
  P99: 52.48 min

03LIC_1071 (962 gaps):
  P25: 0.04 min
  P50: 0.42 min
  P75: 2.35 min
  P90: 8.46 min

03LIC_1016 (375 gaps):
  P25: 0.08 min
  P50: 0.70 min
  P75: 2.58 min
  P90: 9.46 min

03PIC_1013 (2262 gaps):
  P25: 0.02 min
  P50: 0.03 min
  P75: 0.70 min
  P90: 5.63 min

=== Percentage of gaps below candidate thresholds ===
  <=  0.5 min:  63.9% (2299/3599 gaps)
  <=  1.0 min:  71.7% (2580/3599 gaps)
  <=  2.0 min:  79.4% (2857/3599 gaps)
  <=  3.0 min:  83.3% (2999/3599 gaps)
  <=  5.0 min:  87.6% (3151/3599 gaps)
  <= 10.0 min:  92.4% (3324/3599 gaps)


---
## Step 3: Action Sequence Detection and Merging

Based on the time gap analysis above, we define an **action sequence** as consecutive actions on the **same tag** within the **same episode** where each gap is below a configurable threshold.

**Merging logic for each sequence:**
- `sequence_start`: timestamp of the first action
- `sequence_end`: timestamp of the last action
- `initial_value`: `PrevValue` of the first action (starting point)
- `final_value`: `Value` of the last action (ending point)
- `net_magnitude`: `final_value - initial_value` (total cumulative change)
- `net_direction`: 1 if `net_magnitude > 0` else 0
- `num_steps`: count of individual actions in the sequence
- `duration_seconds`: time span from first to last action

We treat a single isolated action as a sequence of length 1 (no merging needed).

In [7]:
def merge_actions_into_sequences(actions_df, threshold_minutes=2.0):
    """
    Merge consecutive actions on the same tag within an episode into composite action sequences.
    
    Parameters:
    - actions_df: DataFrame with columns [episode_id, source, timestamp, value, prev_value, magnitude, description]
    - threshold_minutes: Max time gap (in minutes) between consecutive actions to be grouped
    
    Returns:
    - DataFrame with one row per action sequence (composite action)
    """
    sequences = []
    
    for (ep_id, source), group in actions_df.groupby(['episode_id', 'source']):
        group_sorted = group.sort_values('timestamp').reset_index(drop=True)
        
        if len(group_sorted) == 0:
            continue
        
        # Walk through actions, grouping by time proximity
        current_seq_start_idx = 0
        
        for i in range(1, len(group_sorted) + 1):
            # Check if this action breaks the sequence (or we've reached the end)
            if i == len(group_sorted):
                is_break = True
            else:
                gap_seconds = (group_sorted.loc[i, 'timestamp'] - group_sorted.loc[i-1, 'timestamp']).total_seconds()
                is_break = (gap_seconds / 60.0) > threshold_minutes
            
            if is_break:
                # Finalize the sequence from current_seq_start_idx to i-1
                seq_slice = group_sorted.iloc[current_seq_start_idx:i]
                
                first_action = seq_slice.iloc[0]
                last_action = seq_slice.iloc[-1]
                
                initial_value = first_action['prev_value']
                final_value = last_action['value']
                net_magnitude = final_value - initial_value
                
                # Collect individual magnitudes for the step list
                step_magnitudes = seq_slice['magnitude'].tolist()
                step_timestamps = seq_slice['timestamp'].tolist()
                
                duration_seconds = (last_action['timestamp'] - first_action['timestamp']).total_seconds()
                
                # Determine dominant action type (SP or OP) - most frequent in sequence
                desc_counts = seq_slice['description'].value_counts()
                dominant_type = desc_counts.index[0] if len(desc_counts) > 0 else None
                
                sequences.append({
                    'episode_id': ep_id,
                    'source': source,
                    'sequence_start': first_action['timestamp'],
                    'sequence_end': last_action['timestamp'],
                    'initial_value': initial_value,
                    'final_value': final_value,
                    'net_magnitude': net_magnitude,
                    'net_direction': 1 if net_magnitude > 0 else 0,
                    'num_steps': len(seq_slice),
                    'duration_seconds': duration_seconds,
                    'step_magnitudes': step_magnitudes,
                    'step_timestamps': step_timestamps,
                    'dominant_action_type': dominant_type,
                    'avg_step_magnitude': np.mean(np.abs(step_magnitudes)),
                    'max_step_magnitude': np.max(np.abs(step_magnitudes)),
                })
                
                current_seq_start_idx = i
    
    return pd.DataFrame(sequences)

print("merge_actions_into_sequences() defined.")

merge_actions_into_sequences() defined.


In [8]:
# Run merging with multiple threshold values to compare
thresholds_to_test = [1, 2, 3, 5]
results_by_threshold = {}

for threshold in thresholds_to_test:
    sequences_df = merge_actions_into_sequences(all_actions_df, threshold_minutes=threshold)
    results_by_threshold[threshold] = sequences_df
    
    multi_step = sequences_df[sequences_df['num_steps'] > 1]
    single_step = sequences_df[sequences_df['num_steps'] == 1]
    
    print(f"\n{'='*60}")
    print(f"Threshold: {threshold} minutes")
    print(f"{'='*60}")
    print(f"  Total individual actions: {len(all_actions_df)}")
    print(f"  Total sequences after merging: {len(sequences_df)}")
    print(f"  Reduction: {len(all_actions_df) - len(sequences_df)} actions merged ({(1 - len(sequences_df)/len(all_actions_df))*100:.1f}%)")
    print(f"  Single-step sequences: {len(single_step)} ({len(single_step)/len(sequences_df)*100:.1f}%)")
    print(f"  Multi-step sequences: {len(multi_step)} ({len(multi_step)/len(sequences_df)*100:.1f}%)")
    if len(multi_step) > 0:
        print(f"  Multi-step stats:")
        print(f"    Avg steps per sequence: {multi_step['num_steps'].mean():.1f}")
        print(f"    Max steps per sequence: {multi_step['num_steps'].max()}")
        print(f"    Avg duration: {multi_step['duration_seconds'].mean():.0f}s ({multi_step['duration_seconds'].mean()/60:.1f}min)")
        print(f"    Avg |net magnitude|: {multi_step['net_magnitude'].abs().mean():.2f}")
        print(f"    Avg |step magnitude|: {multi_step['avg_step_magnitude'].mean():.2f}")


Threshold: 1 minutes
  Total individual actions: 3854
  Total sequences after merging: 1274
  Reduction: 2580 actions merged (66.9%)
  Single-step sequences: 727 (57.1%)
  Multi-step sequences: 547 (42.9%)
  Multi-step stats:
    Avg steps per sequence: 5.7
    Max steps per sequence: 90
    Avg duration: 42s (0.7min)
    Avg |net magnitude|: 6.86
    Avg |step magnitude|: 2.33

Threshold: 2 minutes
  Total individual actions: 3854
  Total sequences after merging: 997
  Reduction: 2857 actions merged (74.1%)
  Single-step sequences: 497 (49.8%)
  Multi-step sequences: 500 (50.2%)
  Multi-step stats:
    Avg steps per sequence: 6.7
    Max steps per sequence: 92
    Avg duration: 94s (1.6min)
    Avg |net magnitude|: 6.82
    Avg |step magnitude|: 2.41

Threshold: 3 minutes
  Total individual actions: 3854
  Total sequences after merging: 855
  Reduction: 2999 actions merged (77.8%)
  Single-step sequences: 409 (47.8%)
  Multi-step sequences: 446 (52.2%)
  Multi-step stats:
    Avg ste

In [20]:
# Visualize: Individual vs Composite magnitude distributions
# Use 2-minute threshold as the working example (can be adjusted after review)
working_threshold = 2  # minutes
sequences_df = results_by_threshold[working_threshold]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'Individual Action Magnitudes',
        f'Composite Sequence Net Magnitudes (threshold={working_threshold}min)',
        'Number of Steps per Sequence',
        'Net Magnitude vs Num Steps'
    ]
)

# Individual magnitudes
fig.add_trace(
    go.Histogram(x=all_actions_df['magnitude'], nbinsx=100, name='Individual', marker_color='steelblue'),
    row=1, col=1
)

# Composite net magnitudes
fig.add_trace(
    go.Histogram(x=sequences_df['net_magnitude'], nbinsx=100, name='Composite', marker_color='coral'),
    row=1, col=2
)

# Steps per sequence
multi_step = sequences_df[sequences_df['num_steps'] > 1]
fig.add_trace(
    go.Histogram(x=sequences_df['num_steps'], name='Steps/Sequence', marker_color='green'),
    row=2, col=1
)

# Scatter: net magnitude vs num_steps
fig.add_trace(
    go.Scatter(
        x=multi_step['num_steps'], y=multi_step['net_magnitude'].abs(),
        mode='markers', name='|Net Mag| vs Steps',
        marker=dict(
            color=multi_step['source'].map({'03LIC_1071': 'blue', '03LIC_1016': 'green', '03PIC_1013': 'orange'}),
            size=5, opacity=0.6
        ),
        text=multi_step['source'],
        hovertemplate='Tag: %{text}<br>Steps: %{x}<br>|Net Mag|: %{y:.2f}'
    ),
    row=2, col=2
)

fig.update_xaxes(title_text="Magnitude", row=1, col=1)
fig.update_xaxes(title_text="Net Magnitude", row=1, col=2)
fig.update_xaxes(title_text="Number of Steps", row=2, col=1)
fig.update_xaxes(title_text="Number of Steps", row=2, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_yaxes(title_text="Count", row=2, col=1)
fig.update_yaxes(title_text="|Net Magnitude|", row=2, col=2)

fig.update_layout(height=700, title_text=f"Individual Actions vs Composite Sequences (threshold={working_threshold}min)")
fig.show()

---
## Step 4: Visualize Individual vs Composite Actions on Specific Episodes

Let's pick episodes that have multi-step action sequences and visualize:
- The PV trend
- Individual actions (original)
- Merged composite actions (after merging)

This shows how the operator gradually changed values and what the "full intent" was.

In [10]:
# Find episodes with the most interesting multi-step sequences (for visualization)
multi_step_seqs = sequences_df[sequences_df['num_steps'] > 1].copy()
multi_step_seqs_sorted = multi_step_seqs.sort_values('num_steps', ascending=False)

print("Top 20 longest multi-step action sequences:")
print(multi_step_seqs_sorted[['episode_id', 'source', 'num_steps', 'net_magnitude', 
                               'duration_seconds', 'initial_value', 'final_value']].head(20).to_string())

Top 20 longest multi-step action sequences:
     episode_id      source  num_steps  net_magnitude  duration_seconds  initial_value  final_value
596         443  03PIC_1013         92         6.9999          493.5339        49.0000      55.9999
602         444  03PIC_1013         92         6.9999          493.5339        49.0000      55.9999
599         443  03PIC_1013         57         5.6999          360.1001        49.0000      54.6999
430         393  03PIC_1013         57       -11.4999          338.5501        69.4508      57.9509
605         444  03PIC_1013         57         5.6999          360.1001        49.0000      54.6999
587         442  03PIC_1013         51         6.9999          379.5002        58.0000      64.9999
585         442  03PIC_1013         49         6.8000          262.2831        51.9000      58.7000
160         139  03LIC_1016         39        15.3585          867.0001        29.6415      45.0000
960         601  03PIC_1013         31        -3.1000   

In [11]:
# Visualize a few example episodes: Individual actions vs merged composite actions
# Pick up to 5 episodes with multi-step sequences from different tags

# Select episodes with diverse multi-step sequences
example_episodes = (
    multi_step_seqs_sorted
    .drop_duplicates(subset=['episode_id'])
    .head(5)['episode_id']
    .tolist()
)

for ep_id in example_episodes:
    ep_data = episodes_operated_tags_df[episodes_operated_tags_df['EpisodeID'] == ep_id].iloc[0]
    alarm_start = ep_data['AlarmStart']
    alarm_end = ep_data['AlarmEnd']
    deviation_start = get_deviation_start_for_episode(ep_id, alarm_start, alarm_end)
    
    # Get individual actions for this episode
    ep_individual = all_actions_df[all_actions_df['episode_id'] == ep_id].sort_values('timestamp')
    ep_sequences = sequences_df[sequences_df['episode_id'] == ep_id]
    ep_multi = ep_sequences[ep_sequences['num_steps'] > 1]
    
    # PV window
    pv_start = deviation_start - pd.Timedelta(minutes=10)
    pv_end = alarm_end + pd.Timedelta(minutes=30)
    pv_window = pv_op_data_df.loc[pv_start:pv_end, '03LIC_1071.PV']
    
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            f'Episode {ep_id}: 03LIC_1071.PV with Individual Actions',
            f'Episode {ep_id}: Action Values Over Time (Individual Steps vs Composite)'
        ],
        vertical_spacing=0.15,
        row_heights=[0.5, 0.5]
    )
    
    # Row 1: PV trend with individual action markers
    fig.add_trace(
        go.Scatter(x=pv_window.index, y=pv_window.values, mode='lines', name='03LIC_1071.PV', 
                   line=dict(color='blue')),
        row=1, col=1
    )
    fig.add_hline(y=28.75, line_dash="dash", line_color="red", annotation_text="Alarm Threshold", row=1, col=1)
    fig.add_vrect(x0=alarm_start, x1=alarm_end, fillcolor="red", opacity=0.15, line_width=0, row=1, col=1)
    
    # Mark individual actions on PV
    for source, color in colors.items():
        source_acts = ep_individual[ep_individual['source'] == source]
        if len(source_acts) == 0:
            continue
        act_pv_vals = [get_pv_at_timestamp(t, '03LIC_1071.PV') for t in source_acts['timestamp']]
        fig.add_trace(
            go.Scatter(x=source_acts['timestamp'], y=act_pv_vals, mode='markers',
                       name=f'{source} (individual)', marker=dict(symbol='triangle-up', size=8, color=color),
                       text=[f"Mag: {m:+.2f}" for m in source_acts['magnitude']],
                       hovertemplate='%{text}<br>Time: %{x}'),
            row=1, col=1
        )
    
    # Row 2: Show action values as step chart for each tag
    for source, color in colors.items():
        source_acts = ep_individual[ep_individual['source'] == source].sort_values('timestamp')
        if len(source_acts) == 0:
            continue
        
        # Plot individual steps (prev_value -> value at each timestamp)
        times = []
        values = []
        for _, act in source_acts.iterrows():
            times.extend([act['timestamp'], act['timestamp']])
            values.extend([act['prev_value'], act['value']])
        
        fig.add_trace(
            go.Scatter(x=times, y=values, mode='lines+markers', name=f'{source} steps',
                       line=dict(color=color, width=1), marker=dict(size=4, color=color),
                       opacity=0.6),
            row=2, col=1
        )
        
        # Overlay composite sequence as thick horizontal bars
        source_seqs = ep_sequences[ep_sequences['source'] == source]
        for _, seq in source_seqs.iterrows():
            if seq['num_steps'] > 1:
                # Draw the composite action as a thick arrow/bar
                fig.add_trace(
                    go.Scatter(
                        x=[seq['sequence_start'], seq['sequence_end']],
                        y=[seq['initial_value'], seq['final_value']],
                        mode='lines+markers',
                        line=dict(color=color, width=4, dash='solid'),
                        marker=dict(size=12, symbol='diamond', color=color),
                        name=f'{source} COMPOSITE (net={seq["net_magnitude"]:+.1f}, {seq["num_steps"]} steps)',
                        hovertemplate=f'Net: {seq["net_magnitude"]:+.1f}<br>'
                                      f'Steps: {seq["num_steps"]}<br>'
                                      f'Duration: {seq["duration_seconds"]:.0f}s'
                    ),
                    row=2, col=1
                )
    
    fig.update_xaxes(range=[pv_start, pv_end], row=1, col=1)
    fig.update_xaxes(range=[pv_start, pv_end], row=2, col=1)
    fig.update_yaxes(title_text="Level", row=1, col=1)
    fig.update_yaxes(title_text="Action Value (SP/OP)", row=2, col=1)
    
    fig.update_layout(
        height=700,
        title_text=f'Episode {ep_id}: Individual Actions vs Composite Sequences '
                   f'(threshold={working_threshold}min)<br>'
                   f'<sub>Multi-step sequences: {len(ep_multi)} | '
                   f'Total individual actions: {len(ep_individual)} -> '
                   f'{len(ep_sequences)} sequences</sub>'
    )
    fig.show()

---
## Step 5: Build Composite Action Lookup Table for the Similarity Approach

Now we rebuild the context lookup table using **composite (merged) action sequences** instead of individual actions.

**Key change**: Instead of storing one context row per individual action, we store one context row per **action sequence**. The context is computed at the **start of the sequence** (first action timestamp), and the recommendation includes:
- The net magnitude (total cumulative change)
- The full list of steps (for replay at runtime)
- The direction of the net change

At runtime, when a similar context is matched:
- Instead of recommending a single small step, the system recommends the full composite action
- The runtime system can then **replay the steps** (execute them incrementally over the sequence duration) or **apply the net change directly**

In [12]:
# Build context function (same as similarity approach)
def build_context_for_action(deviation_start, action_timestamp, pv_tags):
    """Build context features for an operator action."""
    context = {}
    for pv_tag in pv_tags:
        tag_name = pv_tag.replace('.PV', '')
        pv_at_deviation = get_pv_at_timestamp(deviation_start, pv_tag)
        pv_at_action = get_pv_at_timestamp(action_timestamp, pv_tag)
        if pd.notna(pv_at_deviation) and pd.notna(pv_at_action) and pv_at_deviation != 0:
            roc_percent = ((pv_at_action - pv_at_deviation) / pv_at_deviation) * 100
        else:
            roc_percent = np.nan
        roc_direction = (1 if roc_percent >= 0 else 0) if pd.notna(roc_percent) else np.nan
        context[f'{tag_name}_pv_at_deviation_start'] = pv_at_deviation
        context[f'{tag_name}_pv_at_action'] = pv_at_action
        context[f'{tag_name}_roc_percent'] = roc_percent
        context[f'{tag_name}_roc_direction'] = roc_direction
    return context

print("build_context_for_action() defined.")

build_context_for_action() defined.


In [13]:
# Build the COMPOSITE context lookup table from no-deviation training episodes
# Step 1: Get training episodes (same split as similarity approach notebook)
no_deviation_episodes_df = episodes_operated_tags_df[
    episodes_operated_tags_df['EpisodeID'].isin(no_deviation_episode_ids)
].copy()

episodes_2025 = episodes_operated_tags_df[
    (episodes_operated_tags_df['AlarmStart'] >= pd.Timestamp('2025-01-01')) &
    (episodes_operated_tags_df['AlarmStart'] < pd.Timestamp('2026-01-01'))
].copy()

np.random.seed(42)
sample_n = 50
if len(episodes_2025) >= sample_n:
    test_episode_ids = episodes_2025.sample(n=sample_n, random_state=42)['EpisodeID'].tolist()
else:
    test_episode_ids = episodes_2025['EpisodeID'].tolist()

train_episodes_df = no_deviation_episodes_df[
    ~no_deviation_episodes_df['EpisodeID'].isin(test_episode_ids)
].copy()

print(f"Training episodes (no deviation, excl. test): {len(train_episodes_df)}")
print(f"Test episodes (2025): {len(test_episode_ids)}")

Training episodes (no deviation, excl. test): 369
Test episodes (2025): 50


In [14]:
# Step 2: Collect individual actions for training episodes only
train_action_records = []
target_sources = ['03LIC_1071', '03LIC_1016', '03PIC_1013']

for _, episode in tqdm(train_episodes_df.iterrows(), total=len(train_episodes_df), desc="Collecting training actions"):
    episode_id = episode['EpisodeID']
    alarm_start = episode['AlarmStart']
    alarm_end = episode['AlarmEnd']
    
    actions, deviation_start = get_operator_actions_for_episode(alarm_start, alarm_end, target_sources)
    if len(actions) == 0:
        continue
    
    actions_clean = actions.dropna(subset=['PrevValue']).copy()
    actions_clean = actions_clean.drop_duplicates(subset=['VT_Start', 'Source', 'Value'])
    actions_clean['Value_num'] = pd.to_numeric(actions_clean['Value'], errors='coerce')
    actions_clean['PrevValue_num'] = pd.to_numeric(actions_clean['PrevValue'], errors='coerce')
    actions_clean = actions_clean.dropna(subset=['Value_num', 'PrevValue_num'])
    actions_clean['magnitude'] = actions_clean['Value_num'] - actions_clean['PrevValue_num']
    
    for source in target_sources:
        source_actions = actions_clean[actions_clean['Source'] == source].sort_values('VT_Start')
        for i, (_, act) in enumerate(source_actions.iterrows()):
            train_action_records.append({
                'episode_id': episode_id,
                'source': source,
                'timestamp': act['VT_Start'],
                'value': act['Value_num'],
                'prev_value': act['PrevValue_num'],
                'magnitude': act['magnitude'],
                'description': act.get('Description', None),
            })

train_actions_df = pd.DataFrame(train_action_records)
print(f"Total individual training actions: {len(train_actions_df)}")

# Step 3: Merge into composite sequences
train_sequences_df = merge_actions_into_sequences(train_actions_df, threshold_minutes=working_threshold)
print(f"Total composite sequences: {len(train_sequences_df)}")
print(f"Multi-step sequences: {(train_sequences_df['num_steps'] > 1).sum()}")
print(f"Single-step sequences: {(train_sequences_df['num_steps'] == 1).sum()}")

Total individual training actions: 941
Total composite sequences: 234
Multi-step sequences: 110
Single-step sequences: 124


In [15]:
# Step 4: Build context for each composite sequence (context at sequence start time)
composite_context_records = []

for _, episode in tqdm(train_episodes_df.iterrows(), total=len(train_episodes_df), desc="Building composite context"):
    episode_id = episode['EpisodeID']
    alarm_start = episode['AlarmStart']
    alarm_end = episode['AlarmEnd']
    deviation_start = get_deviation_start_for_episode(episode_id, alarm_start, alarm_end)
    
    # Get composite sequences for this episode
    ep_seqs = train_sequences_df[train_sequences_df['episode_id'] == episode_id]
    if len(ep_seqs) == 0:
        continue
    
    for _, seq in ep_seqs.iterrows():
        # Build context at the START of the sequence (same as before, but now represents the whole sequence)
        context = build_context_for_action(deviation_start, seq['sequence_start'], context_pv_tags)
        
        # Add composite action metadata
        context['episode_id'] = episode_id
        context['alarm_start'] = alarm_start
        context['alarm_end'] = alarm_end
        context['deviation_start'] = deviation_start
        context['action_timestamp'] = seq['sequence_start']  # Context computed at sequence start
        context['action_source'] = seq['source']
        context['action_value'] = seq['final_value']
        context['action_prev_value'] = seq['initial_value']
        context['action_magnitude'] = seq['net_magnitude']  # NET magnitude (full intent)
        context['action_direction'] = seq['net_direction']
        
        # Additional composite metadata
        context['num_steps'] = seq['num_steps']
        context['sequence_duration_seconds'] = seq['duration_seconds']
        context['avg_step_magnitude'] = seq['avg_step_magnitude']
        context['step_magnitudes'] = str(seq['step_magnitudes'])  # Store as string for CSV
        
        composite_context_records.append(context)

composite_context_df = pd.DataFrame(composite_context_records)
print(f"\nComposite context DataFrame shape: {composite_context_df.shape}")
print(f"Multi-step entries: {(composite_context_df['num_steps'] > 1).sum()}")
print(f"Single-step entries: {(composite_context_df['num_steps'] == 1).sum()}")
print(f"\nActions by source:")
print(composite_context_df['action_source'].value_counts())

Building composite context: 100%|██████████| 369/369 [00:15<00:00, 24.22it/s] 


Composite context DataFrame shape: (234, 126)
Multi-step entries: 110
Single-step entries: 124

Actions by source:
action_source
03PIC_1013    181
03LIC_1071     35
03LIC_1016     18
Name: count, dtype: int64


### Compare: Original (Individual) vs Composite Lookup Tables

Let's compare the magnitude distributions and table sizes.

In [16]:
# Load the original individual-action context table for comparison
original_context_df = pd.read_csv(
    '/home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes/similarity_context_training_no_deviation.csv'
)

print("=== Lookup Table Comparison ===\n")
print(f"{'Metric':<40} {'Individual':>12} {'Composite':>12}")
print(f"{'-'*64}")
print(f"{'Total entries':<40} {len(original_context_df):>12} {len(composite_context_df):>12}")
print(f"{'Unique episodes':<40} {original_context_df['episode_id'].nunique():>12} {composite_context_df['episode_id'].nunique():>12}")

for source in target_sources:
    orig_count = (original_context_df['action_source'] == source).sum()
    comp_count = (composite_context_df['action_source'] == source).sum()
    print(f"{'  ' + source + ' entries':<40} {orig_count:>12} {comp_count:>12}")

print(f"\n{'|Magnitude| statistics':<40} {'Individual':>12} {'Composite':>12}")
print(f"{'-'*64}")
for stat_name, stat_func in [('Mean', 'mean'), ('Median', 'median'), ('Std', 'std'), ('Max', 'max')]:
    orig_val = original_context_df['action_magnitude'].abs().agg(stat_func)
    comp_val = composite_context_df['action_magnitude'].abs().agg(stat_func)
    print(f"{'  ' + stat_name:<40} {orig_val:>12.2f} {comp_val:>12.2f}")

# Per-source magnitude comparison
print(f"\n{'Per-source |Net Magnitude| mean':<40} {'Individual':>12} {'Composite':>12}")
print(f"{'-'*64}")
for source in target_sources:
    orig_val = original_context_df[original_context_df['action_source'] == source]['action_magnitude'].abs().mean()
    comp_val = composite_context_df[composite_context_df['action_source'] == source]['action_magnitude'].abs().mean()
    if pd.notna(orig_val) and pd.notna(comp_val):
        print(f"{'  ' + source:<40} {orig_val:>12.2f} {comp_val:>12.2f}")

=== Lookup Table Comparison ===

Metric                                     Individual    Composite
----------------------------------------------------------------
Total entries                                     941          234
Unique episodes                                    86           86
  03LIC_1071 entries                               91           35
  03LIC_1016 entries                               22           18
  03PIC_1013 entries                              828          181

|Magnitude| statistics                     Individual    Composite
----------------------------------------------------------------
  Mean                                           1.07         3.12
  Median                                         0.10         2.00
  Std                                            2.61         3.48
  Max                                           50.00        34.00

Per-source |Net Magnitude| mean            Individual    Composite
-------------------------------

In [17]:
# Visual comparison: magnitude distributions
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Individual Action Magnitudes (Original Lookup)',
        f'Composite Sequence Net Magnitudes (threshold={working_threshold}min)'
    ]
)

fig.add_trace(
    go.Histogram(x=original_context_df['action_magnitude'], nbinsx=80, 
                 name='Individual', marker_color='steelblue'),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=composite_context_df['action_magnitude'], nbinsx=80,
                 name='Composite', marker_color='coral'),
    row=1, col=2
)

fig.update_xaxes(title_text="Magnitude", row=1, col=1)
fig.update_xaxes(title_text="Net Magnitude", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.update_layout(
    height=400, 
    title_text=f"Magnitude Distribution: Individual Actions vs Composite Sequences<br>"
               f"<sub>Individual: {len(original_context_df)} entries | "
               f"Composite: {len(composite_context_df)} entries</sub>"
)
fig.show()

In [18]:
# Save the composite context lookup table
import os

output_dir = '/home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes'
os.makedirs(output_dir, exist_ok=True)

composite_output_path = f'{output_dir}/similarity_context_training_composite_threshold_{working_threshold}min.csv'
composite_context_df.to_csv(composite_output_path, index=False)
print(f"Composite context table saved to: {composite_output_path}")
print(f"  Shape: {composite_context_df.shape}")
print(f"  Multi-step sequences: {(composite_context_df['num_steps'] > 1).sum()}")
print(f"  Single-step sequences: {(composite_context_df['num_steps'] == 1).sum()}")

Composite context table saved to: /home/h604827/ControlActions/RESULTS/similarity_test_results/no_deviation_episodes/similarity_context_training_composite_threshold_2min.csv
  Shape: (234, 126)
  Multi-step sequences: 110
  Single-step sequences: 124


---
## Summary and Findings

### What was done
1. **Temporal analysis**: Analyzed the distribution of time gaps between consecutive operator actions on the same tag within alarm episodes
2. **Action sequence detection**: Implemented a configurable threshold-based algorithm that groups closely-timed actions into composite sequences
3. **Merging**: For each sequence, computed the net magnitude (final_value - initial_prev_value), capturing the operator's full intent
4. **Lookup table**: Built a new composite-action context table that can replace the original individual-action table in the similarity approach

### Key insight
When operators take incremental actions (e.g., multiple small SP changes within 2 minutes), the original approach stores each small step as an independent action in the lookup table. At runtime, the system would recommend just one small step instead of the full cumulative change. The composite approach captures the **full intended change** as a single entry.

### How to use at runtime
When the similarity approach finds a match in the composite lookup table:
1. **Recommended net magnitude**: Apply the full cumulative change (e.g., +10 instead of +2)
2. **Step replay** (optional): The `step_magnitudes` column stores the individual steps, so the runtime system could replay them incrementally over `sequence_duration_seconds` for smoother execution
3. **Direction**: Based on the net change direction, not individual step directions

### Threshold selection
The `working_threshold` variable controls how aggressively actions are merged. Run the analysis cells above to compare different thresholds (1, 2, 3, 5 minutes) and pick the one that best matches the data patterns. The CDF and quantile analysis in Step 2 will guide this choice.